# LangChain 教程综合复现（阿里云百炼版）

这是一份已经整理好的 LangChain 综合复现手册，按“先基础、后 RAG、再 Tools/Agents、最后扩展”的顺序组织。
运行建议：

1. 按顺序执行，不要跳节。
2. 需要联网的单元都会消耗百炼额度，请先确认账号余额和模型权限。
3. 如果你已经把 `DASHSCOPE_API_KEY` 存到了 Windows 环境变量，但当前 Notebook 内核看不到，这份 Notebook 会自动尝试从 `User/Machine` 环境变量同步到当前进程。

## 00.01 安装依赖

- 建议先直接运行这一格，把本 Notebook 需要的依赖一次装好。
- 如果你之前装过，大多数包会显示已满足，不影响继续运行。

In [1]:
%pip install -U langchain langchain-core langchain-community langchain-classic langchain-text-splitters langchain-chroma chromadb dashscope pypdf nbformat langchain-experimental

  Using cached langchain_experimental-0.4.1-py3-none-any.whl.metadata (1.3 kB)
Using cached langchain_experimental-0.4.1-py3-none-any.whl (210 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 00.02 先改这里的参数

- 这一格最上面就是你最可能会修改的参数。
- 建议先确认 `DASHSCOPE_API_KEY`、模型名和项目路径，再往下运行。

In [ ]:
from __future__ import annotations

import importlib.metadata as importlib_metadata
import os
import subprocess
import sys
import time
import uuid
from pathlib import Path
from typing import Iterator

from langchain_community.chat_models import ChatTongyi
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_community.llms import Tongyi

# ====== 你最可能会改这里 ======
DASHSCOPE_API_KEY_VALUE = os.getenv("DASHSCOPE_API_KEY", "")
CHAT_MODEL_NAME = "qwen-plus"
LLM_MODEL_NAME = "qwen-plus"
EMBEDDING_MODEL_NAME = "text-embedding-v3"
PROJECT_DIR = Path.cwd()
# ============================


def bootstrap_dashscope_api_key() -> tuple[bool, str]:
    process_value = DASHSCOPE_API_KEY_VALUE or os.getenv("DASHSCOPE_API_KEY")
    if process_value:
        os.environ["DASHSCOPE_API_KEY"] = process_value
        return True, "process"

    if os.name == "nt":
        for scope in ("User", "Machine"):
            result = subprocess.run(
                [
                    "powershell",
                    "-NoProfile",
                    "-Command",
                    f"[Environment]::GetEnvironmentVariable('DASHSCOPE_API_KEY', '{scope}')",
                ],
                capture_output=True,
                text=True,
                check=False,
            )
            candidate = result.stdout.strip()
            if candidate:
                os.environ["DASHSCOPE_API_KEY"] = candidate
                return True, f"windows-{scope.lower()}"

    return False, "missing"


def locate_project_root() -> Path:
    signatures = [("examples/rag.txt", "examples/sql.pdf")]
    seen: set[Path] = set()
    candidates: list[Path] = []

    def add_candidate(path: Path) -> None:
        resolved = path.resolve()
        if resolved not in seen and resolved.exists() and resolved.is_dir():
            seen.add(resolved)
            candidates.append(resolved)

    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        add_candidate(base)
        try:
            child_dirs = [child for child in base.iterdir() if child.is_dir()]
        except PermissionError:
            child_dirs = []
        for child in child_dirs:
            add_candidate(child)
            try:
                grandchild_dirs = [grandchild for grandchild in child.iterdir() if grandchild.is_dir()]
            except PermissionError:
                grandchild_dirs = []
            for grandchild in grandchild_dirs:
                add_candidate(grandchild)

    for candidate in candidates:
        if all((candidate / rel).exists() for rel in signatures[0]):
            return candidate

    raise FileNotFoundError("没有找到 01_github_langchain_tutorial 目录，请确认从仓库目录中打开 Notebook。")


def section_ready(name: str, condition: bool, reason: str) -> bool:
    if condition:
        return True
    print(f"[{name}] 跳过：{reason}")
    return False


HAS_DASHSCOPE, KEY_SOURCE = bootstrap_dashscope_api_key()
PROJECT_DIR = locate_project_root()
os.chdir(PROJECT_DIR)
EXAMPLES_DIR = PROJECT_DIR / "examples"
CACHE_DIR = PROJECT_DIR / "cache" / "combined_notebook"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
DASHSCOPE_API_KEY_VALUE = os.environ.get("DASHSCOPE_API_KEY", DASHSCOPE_API_KEY_VALUE)


def get_chat_model(temperature: float = 0.1, streaming: bool = False) -> ChatTongyi:
    return ChatTongyi(
        model=CHAT_MODEL_NAME,
        temperature=temperature,
        streaming=streaming,
        api_key=DASHSCOPE_API_KEY_VALUE,
    )


def get_llm(temperature: float = 0.1, streaming: bool = False) -> Tongyi:
    return Tongyi(
        model=LLM_MODEL_NAME,
        temperature=temperature,
        streaming=streaming,
        api_key=DASHSCOPE_API_KEY_VALUE,
    )


def get_embeddings() -> DashScopeEmbeddings:
    return DashScopeEmbeddings(
        model=EMBEDDING_MODEL_NAME,
        dashscope_api_key=DASHSCOPE_API_KEY_VALUE,
    )


def safe_preview(text: str, limit: int = 120) -> str:
    preview = text[:limit]
    encoding = getattr(getattr(sys, "stdout", None), "encoding", None) or "utf-8"
    return preview.encode(encoding, errors="replace").decode(encoding, errors="replace")


print(f"PROJECT_DIR = {PROJECT_DIR}")
print(f"EXAMPLES_DIR = {EXAMPLES_DIR}")
print(f"CACHE_DIR   = {CACHE_DIR}")
print(f"DASHSCOPE_API_KEY 可用 = {HAS_DASHSCOPE}（来源：{KEY_SOURCE}）")
print(f"CHAT_MODEL_NAME = {CHAT_MODEL_NAME}")
print(f"LLM_MODEL_NAME  = {LLM_MODEL_NAME}")
print(f"EMBEDDING_MODEL = {EMBEDDING_MODEL_NAME}")

if not HAS_DASHSCOPE:
    print("\n如果你还没有设置百炼 Key，可在 PowerShell 里执行：")
    print('$env:DASHSCOPE_API_KEY = "你的阿里云百炼Key"')
    print('[Environment]::SetEnvironmentVariable("DASHSCOPE_API_KEY", "你的阿里云百炼Key", "User")')
    print("设置完用户级环境变量后，请重启 Notebook 内核或重新打开终端。")


PROJECT_DIR = D:\Agent\02-Langchain_simple\langchain_tutorial
EXAMPLES_DIR = D:\Agent\02-Langchain_simple\langchain_tutorial\examples
CACHE_DIR   = D:\Agent\02-Langchain_simple\langchain_tutorial\cache\combined_notebook
DASHSCOPE_API_KEY 可用 = True（来源：windows-user）
CHAT_MODEL_NAME = qwen-plus
LLM_MODEL_NAME  = qwen-plus
EMBEDDING_MODEL = text-embedding-v3


## 00.03 版本摘要

- 先把本次复现依赖的关键版本打印出来。
- 如果你后面遇到行为差异，优先对照这里的版本号。

In [3]:
version_packages = [
    "langchain",
    "langchain-core",
    "langchain-community",
    "langchain-classic",
    "langchain-text-splitters",
    "langchain-chroma",
    "chromadb",
    "dashscope",
    "pypdf",
]

version_table = {}
for package_name in version_packages:
    try:
        version_table[package_name] = importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        version_table[package_name] = "missing"

version_table

{'langchain': '1.2.14',
 'langchain-core': '1.2.24',
 'langchain-community': '0.4.1',
 'langchain-classic': '1.0.3',
 'langchain-text-splitters': '1.1.1',
 'langchain-chroma': '1.1.0',
 'chromadb': '1.5.5',
 'dashscope': '1.25.15',
 'pypdf': '6.9.2'}

## 01.01 PromptTemplate 基础用法

- 先用最基础的 `PromptTemplate` 组织变量，再交给百炼文本模型执行。

In [4]:
from langchain_core.prompts import PromptTemplate

basic_prompt = PromptTemplate.from_template(
    "请用不超过80字解释：{topic}。最后再给一个和 {scene} 相关的例子。"
)

prompt_value = basic_prompt.invoke(
    {
        "topic": "LangChain 中的 PromptTemplate",
        "scene": "RAG 应用",
    }
)

print("格式化后的提示词：")
print(prompt_value.to_string())

if section_ready("01.01", HAS_DASHSCOPE, "未检测到 DASHSCOPE_API_KEY"):
    print("\n模型输出：")
    print(get_llm(temperature=0).invoke(prompt_value.to_string()))

格式化后的提示词：
请用不超过80字解释：LangChain 中的 PromptTemplate。最后再给一个和 RAG 应用 相关的例子。

模型输出：


PromptTemplate 是 LangChain 中用于动态生成提示词的模板类，支持变量插值与格式化。  
例（RAG）：`PromptTemplate.from_template("基于{context}回答：{question}")`，将检索到的文档片段注入 context 变量。


## 01.02 ChatPromptTemplate 与 MessagesPlaceholder

- 这一节演示聊天消息模板和历史消息占位符。
- 这也是后面做多轮对话、Agent、RAG 问答的基础写法。

In [5]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

chat_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个擅长把概念讲清楚的中文老师。"),
        MessagesPlaceholder("history"),
        ("human", "{question}"),
    ]
)

chat_prompt_value = chat_prompt.invoke(
    {
        "history": [
            HumanMessage(content="我在学 LangChain。"),
            AIMessage(content="很好，我们可以先从 Prompt 和 Model 接入开始。"),
        ],
        "question": "那 PromptTemplate 和 ChatPromptTemplate 的区别是什么？",
    }
)

print("渲染后的消息列表：")
for message in chat_prompt_value.messages:
    print(f"- {message.type}: {message.content}")

if section_ready("01.02", HAS_DASHSCOPE, "未检测到 DASHSCOPE_API_KEY"):
    response = get_chat_model(temperature=0).invoke(chat_prompt_value.messages)
    print("\n模型输出：")
    print(response.content)

渲染后的消息列表：
- system: 你是一个擅长把概念讲清楚的中文老师。
- human: 我在学 LangChain。
- ai: 很好，我们可以先从 Prompt 和 Model 接入开始。
- human: 那 PromptTemplate 和 ChatPromptTemplate 的区别是什么？



模型输出：
这是 LangChain 中非常关键、也常被初学者混淆的一对概念。我们来用「讲清楚 + 举例子 + 记忆口诀」的方式帮你彻底理解：

---

🔹 **核心区别一句话总结**：  
> `PromptTemplate` 是为 **普通文本模型（LLM）** 设计的，输入是纯文本 prompt；  
> `ChatPromptTemplate` 是为 **聊天模型（ChatModel）** 设计的，输入是结构化的「消息序列」（如 `SystemMessage`, `HumanMessage`, `AIMessage`）。

---

🔹 **为什么需要区分？—— 模型接口不同**

| 类型 | 典型模型示例 | 输入格式要求 | LangChain 对应类 |
|------|----------------|----------------|-------------------|
| 文本大模型（LLM） | `OpenAI(model="gpt-3.5-turbo-instruct")`、`Ollama(model="llama3")` | 一个连续字符串（如 `"你是一个翻译助手。请把以下英文翻译成中文：{text}"`） | `PromptTemplate` |
| 聊天大模型（ChatModel） | `ChatOpenAI(model="gpt-4o")`、`ChatOllama(model="llama3")` | 一组带角色的消息列表，例如 `[SystemMessage(...), HumanMessage(...), AIMessage(...)]` | `ChatPromptTemplate` |

⚠️ 注意：即使都用 `gpt-4o`，如果你用的是 `ChatOpenAI`（推荐），就必须用 `ChatPromptTemplate`；如果误用 `PromptTemplate` + `ChatOpenAI`，会报错或行为异常（比如把 system message 当作普通文本拼接，失去角色语义）。

---

🔹 **代码对比（一目了然）**

✅ 正确用法：

```python
# ✅ 场景1：用 ChatOpenAI → 必须用 ChatPromptTemplate
from langchain_core.prompts

## 02.01 Tongyi 文本模型：invoke 与 batch

- 先看单次调用，再看批量调用。

In [6]:
if section_ready("02.01", HAS_DASHSCOPE, "未检测到 DASHSCOPE_API_KEY"):
    llm = get_llm(temperature=0)

    single_answer = llm.invoke("请用一句话解释什么是向量数据库。")
    print("单次调用：")
    print(single_answer)

    batch_answers = llm.batch(
        [
            "请用一句话解释什么是 RAG。",
            "请用一句话解释什么是 Embedding。",
        ]
    )
    print("\n批量调用：")
    for index, answer in enumerate(batch_answers, start=1):
        print(f"[{index}] {answer}")

单次调用：
向量数据库是一种专门用于高效存储、索引和检索高维向量（如由AI模型生成的语义嵌入）的数据库系统，支持基于相似度（如余弦相似度、欧氏距离）的近似最近邻（ANN）搜索，广泛应用于语义搜索、推荐系统和大模型RAG等场景。



批量调用：
[1] RAG（Retrieval-Augmented Generation，检索增强生成）是一种将外部知识检索与大语言模型生成能力相结合的技术：在回答问题前，先从可靠的知识库中检索相关文档片段，再将这些片段作为上下文输入给大语言模型，从而生成更准确、可溯源、且时效性更强的回答。
[2] Embedding 是一种将离散、高维、稀疏的符号（如单词、用户、物品等）映射为连续、低维、稠密的实数向量的技术，使得语义或结构相似的符号在向量空间中距离更近，便于机器学习模型处理和计算。


## 02.02 ChatTongyi：消息式调用、流式输出与 token 使用量

- 先做一次标准聊天调用，再演示 `stream()` 的逐块输出。

In [7]:
if section_ready("02.02", HAS_DASHSCOPE, "未检测到 DASHSCOPE_API_KEY"):
    chat = get_chat_model(temperature=0)
    messages = [
        SystemMessage(content="你是一个简洁的中文助手。"),
        HumanMessage(content="请用三点说明 LangChain 在 RAG 里的作用。"),
    ]

    response = chat.invoke(messages)
    print("聊天模型输出：")
    print(response.content)
    print("\nresponse_metadata：")
    print(response.response_metadata)

    print("\n流式输出：")
    streaming_chat = get_chat_model(temperature=0, streaming=True)
    for chunk in streaming_chat.stream(messages):
        if chunk.content:
            print(chunk.content, end="", flush=True)
    print()

聊天模型输出：
1. **编排与集成**：LangChain 提供统一接口，便捷集成文档加载、文本分割、向量存储（如 Chroma）、大模型调用等 RAG 各环节组件，降低工程耦合度。

2. **检索增强编排**：内置 `RetrievalQA`、`ConversationalRetrievalChain` 等链式模块，自动完成“用户查询 → 向量检索 → 上下文注入 → LLM 生成”全流程，支持重排序、元数据过滤等增强策略。

3. **可扩展性与调试支持**：通过 `Runnable` 抽象和回调机制（CallbackHandler），便于插入自定义逻辑（如检索优化、结果验证）、记录中间步骤（如检索到的 chunk），提升 RAG 的可观测性与迭代效率。

response_metadata：
{'model_name': 'qwen-plus', 'finish_reason': 'stop', 'request_id': '1589f07c-c3bc-914c-8a06-e2106736c036', 'token_usage': {'input_tokens': 33, 'output_tokens': 176, 'total_tokens': 209, 'prompt_tokens_details': {'cached_tokens': 0}}}

流式输出：


1

. **编

排与集成

**：Lang

...

AG 行为（

如混合检索、元

数据过滤、结果

验证），提升准确性

与可控性。

## 02.03 LLM 缓存

- 第一次调用会真正访问模型，第二次命中内存缓存通常会更快。

In [8]:
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache

if section_ready("02.03", HAS_DASHSCOPE, "未检测到 DASHSCOPE_API_KEY"):
    cache_prompt = "请用一句话解释什么是检索器。"
    cached_llm = get_llm(temperature=0)

    set_llm_cache(InMemoryCache())

    start = time.perf_counter()
    first_response = cached_llm.invoke(cache_prompt)
    first_cost = time.perf_counter() - start

    start = time.perf_counter()
    second_response = cached_llm.invoke(cache_prompt)
    second_cost = time.perf_counter() - start

    print("第一次调用：", first_response)
    print(f"第一次耗时：{first_cost:.3f}s")
    print("第二次调用：", second_response)
    print(f"第二次耗时：{second_cost:.3f}s")

    set_llm_cache(None)

第一次调用： 检索器是一种从大规模数据集合（如文档、网页或数据库）中根据用户查询快速定位并返回相关结果的系统或算法组件。
第一次耗时：1.437s
第二次调用： 检索器是一种从大规模数据集合（如文档、网页或数据库）中根据用户查询快速定位并返回相关结果的系统或算法组件。
第二次耗时：0.000s


## 03.01 PydanticOutputParser

- 用结构化输出约束模型把答案整理成 Pydantic 对象。

In [9]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser


class ConceptCard(BaseModel):
    concept: str = Field(description="概念名称")
    core_idea: str = Field(description="一句话解释核心思想")
    rag_example: str = Field(description="一个和 RAG 相关的例子")


card_parser = PydanticOutputParser(pydantic_object=ConceptCard)
card_prompt = PromptTemplate(
    template=(
        "请把主题整理成结构化结果。\n"
        "{format_instructions}\n"
        "主题：{topic}"
    ),
    input_variables=["topic"],
    partial_variables={"format_instructions": card_parser.get_format_instructions()},
)

print("格式约束预览：")
print(card_parser.get_format_instructions()[:500] + "...")

if section_ready("03.01", HAS_DASHSCOPE, "未检测到 DASHSCOPE_API_KEY"):
    parser_chain = card_prompt | get_chat_model(temperature=0) | card_parser
    parsed_card = parser_chain.invoke({"topic": "LangChain 在 RAG 系统中的作用"})
    print(parsed_card.model_dump())

格式约束预览：
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"concept": {"description": "概念名称"...


{'concept': 'LangChain 在 RAG 系统中的作用', 'core_idea': 'LangChain 是一个用于构建基于语言模型的应用程序的框架，它在 RAG 系统中提供模块化组件（如文档加载器、文本分割器、向量存储集成、检索器和链式调用），简化了检索增强生成流程的开发与编排。', 'rag_example': '使用 LangChain 的 DocumentLoader 读取 PDF 文档，通过 TextSplitter 分块，嵌入后存入 Chroma 向量数据库；用户提问时，LangChain 的 RetrievalQA 链自动执行相似性检索并将其上下文注入 LLM 提示，生成准确、有依据的回答。'}


## 03.02 JsonOutputParser

- 如果你只需要 JSON 而不是完整的 Pydantic 对象，可以用 `JsonOutputParser`。
- 这个写法在做轻量级配置生成时很常见。

In [10]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_prompt = PromptTemplate(
    template=(
        "请按下面的 JSON 约束输出，不要添加额外说明。\n"
        "{format_instructions}\n"
        "主题：{topic}"
    ),
    input_variables=["topic"],
    partial_variables={"format_instructions": json_parser.get_format_instructions()},
)

if section_ready("03.02", HAS_DASHSCOPE, "未检测到 DASHSCOPE_API_KEY"):
    json_chain = json_prompt | get_chat_model(temperature=0) | json_parser
    json_result = json_chain.invoke({"topic": "RAG 的基本流程"})
    json_result

## 04.01 TextLoader、CSVLoader、PyPDFLoader

- 使用教程目录里的示例文件做统一演示。

In [11]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_community.document_loaders.csv_loader import CSVLoader

text_docs = TextLoader(str(EXAMPLES_DIR / "rag.txt"), encoding="utf-8").load()
csv_docs = CSVLoader(str(EXAMPLES_DIR / "test.csv")).load()
pdf_docs = PyPDFLoader(str(EXAMPLES_DIR / "sql.pdf")).load()

print("TextLoader ->", len(text_docs), "条")
print(safe_preview(text_docs[0].page_content))
print("\nCSVLoader ->", len(csv_docs), "条")
print(safe_preview(csv_docs[0].page_content))
print("\nPyPDFLoader ->", len(pdf_docs), "页")
print(safe_preview(pdf_docs[0].page_content))


Previous trailer cannot be read: ("invalid literal for int() with base 10: b'/Root'",)


Object 14 0 found


Object 3 0 found


Object 2 0 found


Object 5 0 found


...

Object 12 0 found


Object 35 0 found


Object 34 0 found


Object 4 0 found


TextLoader -> 1 条
2024年普通高等学校招生全国统一考试（简称：2024年全国高考），是中华人民共和国合格的高中毕业生或具有同等学力的考生参加的选拔性考试 [1-2]。2024年报名人数1342万人，比2023年增加51万人 [21]。


2024年高考是

CSVLoader -> 2 条
id: 1
name: 张三
degree: 本科

PyPDFLoader -> 1 页
创建表  
插⼊数据  
# 分区表
create table test_t2(words string,frequency string) partitioned by (partdate string) row 
format deli


## 05.01 CharacterTextSplitter 与 RecursiveCharacterTextSplitter

- 先看字符切分，再看更适合中文 RAG 的递归切分。

In [12]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

rag_text = (EXAMPLES_DIR / "rag.txt").read_text(encoding="utf-8")

char_splitter = CharacterTextSplitter(separator="\n", chunk_size=120, chunk_overlap=20)
char_chunks = char_splitter.split_text(rag_text)

recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n\n", "\n\n", "\n", "。", "；", "，", " "],
    chunk_size=220,
    chunk_overlap=40,
    add_start_index=True,
)
recursive_chunks = recursive_splitter.create_documents([rag_text])

print("CharacterTextSplitter 切分块数：", len(char_chunks))
print(char_chunks[0][:120])
print("\nRecursiveCharacterTextSplitter 切分块数：", len(recursive_chunks))
print(recursive_chunks[0])

Created a chunk of size 143, which is longer than the specified 120


Created a chunk of size 124, which is longer than the specified 120


Created a chunk of size 130, which is longer than the specified 120


Created a chunk of size 487, which is longer than the specified 120


Created a chunk of size 163, which is longer than the specified 120


...

Created a chunk of size 156, which is longer than the specified 120


Created a chunk of size 308, which is longer than the specified 120


Created a chunk of size 170, which is longer than the specified 120


Created a chunk of size 141, which is longer than the specified 120


CharacterTextSplitter 切分块数： 40
2024年普通高等学校招生全国统一考试（简称：2024年全国高考），是中华人民共和国合格的高中毕业生或具有同等学力的考生参加的选拔性考试 [1-2]。2024年报名人数1342万人，比2023年增加51万人 [21]。

RecursiveCharacterTextSplitter 切分块数： 33
page_content='2024年普通高等学校招生全国统一考试（简称：2024年全国高考），是中华人民共和国合格的高中毕业生或具有同等学力的考生参加的选拔性考试 [1-2]。2024年报名人数1342万人，比2023年增加51万人 [21]。


2024年高考是黑龙江、甘肃、吉林、安徽、江西、贵州、广西7个省份（中国第四批高考综合改革省份）的第一届落地实施的新高考。 [3]' metadata={'start_index': 0}


## 05.02 代码切分与 Markdown 标题切分

- 这一节把原 notebook 里的代码切分和 Markdown 标题切分保留下来。
- 这两类切分很适合做代码知识库和文档知识库。

In [13]:
from langchain_text_splitters import Language, MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

python_code = '''
def retrieve(query: str) -> list[str]:
    # 根据 query 检索文档
    docs = vector_store.similarity_search(query, k=3)
    return [doc.page_content for doc in docs]

def answer(query: str) -> str:
    contexts = retrieve(query)
    return "\n".join(contexts)
'''

code_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=90,
    chunk_overlap=10,
)
code_chunks = code_splitter.split_text(python_code)

markdown_document = '''
# LangChain 笔记
## Prompt
Prompt 负责组织输入。
## Retriever
Retriever 负责找上下文。
### MultiQuery
可以让模型改写多个检索问题。
'''

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")]
)
markdown_docs = markdown_splitter.split_text(markdown_document)

print("代码切分结果：")
for index, chunk in enumerate(code_chunks, start=1):
    print(f"[{index}] {chunk.strip()}\n")

print("Markdown 标题切分结果：")
markdown_docs

代码切分结果：
[1] def retrieve(query: str) -> list[str]:
    # 根据 query 检索文档

[2] docs = vector_store.similarity_search(query, k=3)

[3] return [doc.page_content for doc in docs]

[4] def answer(query: str) -> str:
    contexts = retrieve(query)
    return "

[5] ".join(contexts)

Markdown 标题切分结果：


[Document(metadata={'h1': 'LangChain 笔记', 'h2': 'Prompt'}, page_content='Prompt 负责组织输入。'),
 Document(metadata={'h1': 'LangChain 笔记', 'h2': 'Retriever'}, page_content='Retriever 负责找上下文。'),
 Document(metadata={'h1': 'LangChain 笔记', 'h2': 'Retriever', 'h3': 'MultiQuery'}, page_content='可以让模型改写多个检索问题。')]

## 06.01 Embedding 与余弦相似度

- 我们用百炼嵌入模型算文档向量，再自己写一个余弦相似度函数观察效果。

In [14]:
import math


def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    return dot / (norm_a * norm_b)


embedding_texts = [
    "LangChain 可以把提示词、模型和工具串起来。",
    "RAG 会先检索文档，再把上下文交给模型回答。",
    "Embedding 可以把文本映射到向量空间。",
]

if section_ready("06.01", HAS_DASHSCOPE, "未检测到 DASHSCOPE_API_KEY"):
    embeddings_model = get_embeddings()
    document_vectors = embeddings_model.embed_documents(embedding_texts)
    query_vector = embeddings_model.embed_query("LangChain 如何支持 RAG？")

    scores = [
        cosine_similarity(query_vector, vector)
        for vector in document_vectors
    ]

    for index, (text, score) in enumerate(zip(embedding_texts, scores), start=1):
        print(f"[{index}] score={score:.4f} | {text}")

[1] score=0.6521 | LangChain 可以把提示词、模型和工具串起来。
[2] score=0.6509 | RAG 会先检索文档，再把上下文交给模型回答。
[3] score=0.4269 | Embedding 可以把文本映射到向量空间。


## 07.01 向量库：Chroma + 相似度检索 + 向量检索 + MMR

- 这一节把建库、相似度检索、向量检索、MMR 放到一起演示。
- 为了避免 Windows 下额外安装 `faiss-cpu` 的麻烦，这里统一用 `Chroma`。

In [15]:
from langchain_community.document_loaders import TextLoader
from langchain_chroma import Chroma

if section_ready("07.01", HAS_DASHSCOPE, "未检测到 DASHSCOPE_API_KEY"):
    rag_raw_documents = TextLoader(
        str(EXAMPLES_DIR / "rag.txt"),
        encoding="utf-8",
    ).load()

    rag_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n\n", "\n\n", "\n", "。", "；", "，", " "],
        chunk_size=220,
        chunk_overlap=40,
        add_start_index=True,
    )
    rag_documents = rag_splitter.split_documents(rag_raw_documents)
    rag_embeddings = get_embeddings()

    rag_vector_store = Chroma(
        collection_name=f"combined_rag_{uuid.uuid4().hex[:8]}",
        embedding_function=rag_embeddings,
        persist_directory=str(CACHE_DIR / "chroma"),
    )
    rag_vector_store.add_documents(rag_documents)

    query = "哪里可以了解高考成绩"

    similarity_docs = rag_vector_store.similarity_search(query, k=2)
    vector_docs = rag_vector_store.similarity_search_by_vector(
        rag_embeddings.embed_query(query),
        k=2,
    )
    mmr_docs = rag_vector_store.max_marginal_relevance_search(query, k=2, fetch_k=6)

    print("similarity_search:")
    print(similarity_docs[0].page_content[:120])
    print("\nsimilarity_search_by_vector:")
    print(vector_docs[0].page_content[:120])
    print("\nmax_marginal_relevance_search:")
    print(mmr_docs[0].page_content[:120])

similarity_search:
一、在哪里可以了解高考成绩、志愿填报时间和方式、各高校招生计划、往年录取参考等志愿填报权威信息？
各省级教育行政部门或招生考试机构官方网站、微信公众号等权威渠道都会公布今年高考各阶段工作时间安排，包括高考成绩公布时间和查询方式、志愿填报时间

similarity_search_by_vector:
一、在哪里可以了解高考成绩、志愿填报时间和方式、各高校招生计划、往年录取参考等志愿填报权威信息？
各省级教育行政部门或招生考试机构官方网站、微信公众号等权威渠道都会公布今年高考各阶段工作时间安排，包括高考成绩公布时间和查询方式、志愿填报时间

max_marginal_relevance_search:
一、在哪里可以了解高考成绩、志愿填报时间和方式、各高校招生计划、往年录取参考等志愿填报权威信息？
各省级教育行政部门或招生考试机构官方网站、微信公众号等权威渠道都会公布今年高考各阶段工作时间安排，包括高考成绩公布时间和查询方式、志愿填报时间


## 08.01 基础检索器与 MultiQueryRetriever

- 先把向量库转成 retriever，再让模型自动生成多个检索问题。

In [16]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

if section_ready("08.01", HAS_DASHSCOPE, "未检测到 DASHSCOPE_API_KEY"):
    rag_retriever = rag_vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 2},
    )
    base_docs = rag_retriever.invoke("哪里可以了解高考成绩")
    print("基础检索返回数：", len(base_docs))

    multi_query_retriever = MultiQueryRetriever.from_llm(
        retriever=rag_retriever,
        llm=get_chat_model(temperature=0),
        include_original=True,
    )
    multi_query_docs = multi_query_retriever.invoke("哪里可以了解高考成绩")
    print("MultiQueryRetriever 返回数：", len(multi_query_docs))
    print(multi_query_docs[0].page_content[:120])

基础检索返回数： 2


MultiQueryRetriever 返回数： 4
一、在哪里可以了解高考成绩、志愿填报时间和方式、各高校招生计划、往年录取参考等志愿填报权威信息？
各省级教育行政部门或招生考试机构官方网站、微信公众号等权威渠道都会公布今年高考各阶段工作时间安排，包括高考成绩公布时间和查询方式、志愿填报时间


## 08.02 ContextualCompressionRetriever

- 这一节演示“先召回，再压缩”的思路。
- 压缩器只保留和问题强相关的片段，适合后面塞给模型回答。

In [17]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

if section_ready("08.02", HAS_DASHSCOPE, "未检测到 DASHSCOPE_API_KEY"):
    compressor = LLMChainExtractor.from_llm(get_chat_model(temperature=0))
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=rag_retriever,
    )

    compressed_docs = compression_retriever.invoke("哪里可以了解高考成绩")
    print("压缩后文档数：", len(compressed_docs))
    print(compressed_docs[0].page_content[:160])

压缩后文档数： 2
各省级教育行政部门或招生考试机构官方网站、微信公众号等权威渠道都会公布今年高考各阶段工作时间安排，包括高考成绩公布时间和查询方式、志愿填报时间，以及今年各高校招生计划、往年录取情况参考等权威信息。


## 09.01 定义 Tool 并查看工具 Schema

- 先用 `@tool` 定义工具，再看它如何被转成 OpenAI 兼容的 schema。

In [18]:
from datetime import datetime

from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool


class WeatherSchema(BaseModel):
    location: str = Field(description="城市名，例如杭州、北京")


@tool("get_current_weather", args_schema=WeatherSchema)
def get_current_weather(location: str) -> str:
    """查询指定城市的天气。"""
    return f"{location}今天晴转多云。"


@tool("get_current_time")
def get_current_time() -> str:
    """查询当前时间。"""
    return datetime.now().strftime("当前时间：%Y-%m-%d %H:%M:%S")


tools = [get_current_weather, get_current_time]
[convert_to_openai_tool(tool_item) for tool_item in tools]

[{'type': 'function',
  'function': {'name': 'get_current_weather',
   'description': '查询指定城市的天气。',
   'parameters': {'properties': {'location': {'description': '城市名，例如杭州、北京',
      'type': 'string'}},
    'required': ['location'],
    'type': 'object'}}},
 {'type': 'function',
  'function': {'name': 'get_current_time',
   'description': '查询当前时间。',
   'parameters': {'properties': {}, 'type': 'object'}}}]

## 09.02 ChatTongyi.bind_tools 手工执行工具调用

- 这一步最适合理解 Tool Calling 的底层流程。
- 先让模型决定要调用哪些工具，再把工具结果回填给模型，让它生成最终回答。

In [19]:
from langchain_core.messages import HumanMessage, ToolMessage

if section_ready("09.02", HAS_DASHSCOPE, "未检测到 DASHSCOPE_API_KEY"):
    tool_map = {
        "get_current_weather": get_current_weather,
        "get_current_time": get_current_time,
    }

    model_with_tools = get_chat_model(temperature=0).bind_tools(tools)
    messages = [HumanMessage(content="杭州和北京天气怎么样？现在几点了？")]

    assistant_output = model_with_tools.invoke(messages)
    print("第一轮模型输出：")
    print(assistant_output)

    while assistant_output.tool_calls:
        messages.append(assistant_output)
        for tool_call in assistant_output.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_result = tool_map[tool_name].invoke(tool_args)
            print(f"工具 {tool_name} 输入：{tool_args}")
            print(f"工具 {tool_name} 输出：{tool_result}")
            print("-" * 60)
            messages.append(
                ToolMessage(
                    content=tool_result,
                    tool_call_id=tool_call["id"],
                    name=tool_name,
                )
            )

        assistant_output = model_with_tools.invoke(messages)

    print("最终回答：")
    print(assistant_output.content)

第一轮模型输出：
content='' additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_dc50ff61d7374438bf7f40', 'type': 'function', 'function': {'name': 'get_current_weather', 'arguments': '{"location": "杭州"}'}}, {'index': 1, 'id': 'call_f06e717660e841789ca36d', 'type': 'function', 'function': {'name': 'get_current_weather', 'arguments': '{"location": "北京"}'}}, {'index': 2, 'id': 'call_ea9ba4edacb447a99d8f14', 'type': 'function', 'function': {'name': 'get_current_time', 'arguments': '{}'}}]} response_metadata={'model_name': 'qwen-plus', 'finish_reason': 'tool_calls', 'request_id': 'dd32e2d3-7e4c-90a3-9ea2-26d29af65062', 'token_usage': {'input_tokens': 208, 'output_tokens': 58, 'total_tokens': 266, 'prompt_tokens_details': {'cached_tokens': 0}}} id='lc_run--019d4e34-e577-7e33-8fcc-55edb80e2bae-0' tool_calls=[{'name': 'get_current_weather', 'args': {'location': '杭州'}, 'id': 'call_dc50ff61d7374438bf7f40', 'type': 'tool_call'}, {'name': 'get_current_weather', 'args': {'location': '北京'}, 'id': 'cal

最终回答：
杭州和北京今天的天气都是晴转多云，当前时间是2026年4月2日20:39:44。


## 09.03 create_agent 快速搭建可调用工具的 Agent

- 在当前版本组合下，`create_agent` + `ChatTongyi` 已验证可运行。

In [20]:
from langchain.agents import create_agent

if section_ready("09.03", HAS_DASHSCOPE, "未检测到 DASHSCOPE_API_KEY"):
    weather_agent = create_agent(
        model=get_chat_model(temperature=0),
        tools=tools,
        system_prompt="你是一个严谨的中文助手，遇到天气或时间问题时要优先调用工具。",
    )

    agent_result = weather_agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "请告诉我杭州天气，再顺带告诉我现在几点。",
                }
            ]
        }
    )

    print("Agent 最终输出：")
    print(agent_result["messages"][-1].content)

Agent 最终输出：
杭州今天天气为晴转多云，当前时间为2026年4月2日20:39:48。


## 10. 扩展章节

- 下面这些内容属于原教程里的进阶主题。
- 我把它们统一放到最后，避免打断前面的主线复现。
- 建议先跑完 `00` 到 `09`，再按需继续跑这里。

## 10.01 Few-shot Prompt

- 适合在你希望模型模仿某种固定回答风格时使用。

In [21]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

examples = [
    {"question": "什么是向量数据库？", "answer": "它用来存储和检索向量，常见于语义搜索和 RAG。"},
    {"question": "什么是检索器？", "answer": "它负责从知识库中找出和问题最相关的内容。"},
]

example_prompt = PromptTemplate.from_template("问题：{question}\n回答：{answer}")
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="问题：{input}\n回答：",
    input_variables=["input"],
    prefix="请参考下面的示例风格回答问题：",
)

rendered = few_shot_prompt.format(input="什么是 Embedding？")
print(rendered)

if section_ready("10.01", HAS_DASHSCOPE, "未检测到 DASHSCOPE_API_KEY"):
    print("\n模型输出：")
    print(get_llm(temperature=0).invoke(rendered))

请参考下面的示例风格回答问题：

问题：什么是向量数据库？
回答：它用来存储和检索向量，常见于语义搜索和 RAG。

问题：什么是检索器？
回答：它负责从知识库中找出和问题最相关的内容。

问题：什么是 Embedding？
回答：

模型输出：


它是一种将文本、图像等数据转换为稠密向量的数学表示，用于捕捉语义信息并支持向量检索。


## 10.02 自定义文档加载器

- 示例保持很小，只做“把 Markdown 文件读成 Document”。

In [22]:
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document


class SqlMarkdownLoader(BaseLoader):
    def __init__(self, path: Path) -> None:
        self.path = Path(path)

    def lazy_load(self) -> Iterator[Document]:
        content = self.path.read_text(encoding="utf-8")
        yield Document(
            page_content=content,
            metadata={
                "source": str(self.path),
                "loader": self.__class__.__name__,
                "line_count": len(content.splitlines()),
            },
        )


custom_docs = list(SqlMarkdownLoader(EXAMPLES_DIR / "sql.md").lazy_load())
custom_docs[0]

Document(metadata={'source': 'D:\\Agent\\02-Langchain_simple\\langchain_tutorial\\examples\\sql.md', 'loader': 'SqlMarkdownLoader', 'line_count': 33}, page_content="## 创建表\n\n```sql\n# 分区表\ncreate table test_t2(words string,frequency string) partitioned by (partdate string) row format delimited fields terminated by ',';\n\n# orc表\nCREATE TABLE IF NOT EXISTS bank.account_orc (\n  `id_card` int,\n  `tran_time` string,\n  `name` string,\n  `cash` int\n  )\nstored as orc;\n```\n\n# 插入数据\n\n```sql\ninsert into tablename values('col1', 'col2');\n\n\nINSERT INTO table_name (column1, column2, column3)\nVALUES\n(value1, value2, value3),\n(value4, value5, value6),\n(value7, value8, value9);\n\n\nINSERT OVERWRITE TABLE tb\nselect * from tb2\n;\n```")

## 10.03 Semantic Chunking（可选）

- 如果本机没有安装 `langchain-experimental`，会给出安装提示并跳过。

In [23]:
if section_ready("10.03", HAS_DASHSCOPE, "未检测到 DASHSCOPE_API_KEY"):
    try:
        from langchain_experimental.text_splitter import SemanticChunker
    except ImportError:
        print("未安装 langchain-experimental，可执行：")
        print("%pip install -U langchain-experimental")
    else:
        semantic_splitter = SemanticChunker(
            embeddings=get_embeddings(),
            breakpoint_threshold_type="percentile",
            breakpoint_threshold_amount=80,
        )
        semantic_docs = semantic_splitter.create_documents([rag_text[:1500]])
        print("语义分块结果数：", len(semantic_docs))
        print(semantic_docs[0])

语义分块结果数： 1
page_content='2024年普通高等学校招生全国统一考试（简称：2024年全国高考），是中华人民共和国合格的高中毕业生或具有同等学力的考生参加的选拔性考试 [1-2]。2024年报名人数1342万人，比2023年增加51万人 [21]。


2024年高考是黑龙江、甘肃、吉林、安徽、江西、贵州、广西7个省份（中国第四批高考综合改革省份）的第一届落地实施的新高考。 [3]


2024年高考全国统考于2024年6月7日开始举行，部分省份考试时间为2天，实行新高考的省份为3-4天 [29]。5月31日，2024年高考试卷从北京发往全国 [22]。6月5日，2024年高考举报电话已开通，教育部教育考试院的举报电话为：010-62790357 [53] [77]。


地区,报名时间
北京,2023年10月25日9时至28日17时（进城务工人员随迁子女申请时间为10月10日9时至11日17时）
上海,2023年10月16日-19日（每天8:00-21:00）、10月20日（8:00-16:00）
天津,2023年11月1日9时至7日17时
重庆,2023年10月24日-11月7日
河北,2023年10月30日09时至11月13日17时
山西,2023年11月5日8:00—10日18:00
内蒙古,2023年11月2日9:00至13日18:00
山东,2023年11月9日至15日（每天9:00—18:00）
江苏,2023年11月1日至3日（8:30-22:00）;11月4日（8:30-17:00）
浙江,2023年11月1日9:00至10日17:00
江西,2023年11月1日9:00—7日17:00
福建,2023年10月25日至30日
安徽,2023年10月25日10:00至29日17:00
河南,艺术类为2023年11月1日9:00至5日17:00；非艺术类为11月8日9:00至23日17:00
湖南,2023年10月23日至31日
湖北,2023年11月8日-18日
四川,2023年10月14日至20日
云南,2023年11月5-15日
贵州,2023年11月1日00:00至10日24:00
西藏,2023年11月1日至12月1日
辽宁,2023年10月27日至10月31日
吉林,10月5日—10日（9:00—16:30）
黑龙江,1

## 10.04 Embedding 缓存

- 思路是把嵌入结果落到本地，第二次重复请求就不用再调接口。

In [24]:
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

if section_ready("10.04", HAS_DASHSCOPE, "未检测到 DASHSCOPE_API_KEY"):
    cache_store = LocalFileStore(str(CACHE_DIR / "embedding_cache"))
    cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
        underlying_embeddings=get_embeddings(),
        document_embedding_cache=cache_store,
        namespace=EMBEDDING_MODEL_NAME,
        key_encoder="sha256",
    )

    cached_once = cached_embeddings.embed_documents(embedding_texts)
    cached_twice = cached_embeddings.embed_documents(embedding_texts)

    print("第一次向量数：", len(cached_once))
    print("第二次向量数：", len(cached_twice))
    print("缓存 key 示例：", list(cache_store.yield_keys())[:3])

第一次向量数： 3
第二次向量数： 3
缓存 key 示例： ['text-embedding-v3002f30c450984b70da55b970ec65533de1e2a62ac99e4b6afeea0604d9571649', 'text-embedding-v3a3fd29ee3d8fa9d09198243865db95cd420b60d487b7ee2e409886555f9fc2dc', 'text-embedding-v3e84da76695ebedf1f332c28dbe67b9072bb821a001ce869041f2a2d3ac8cd167']


## 10.05 ParentDocumentRetriever

- 用小块做召回，用大块回传内容，是 RAG 里很常见的进阶技巧。

In [25]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore

if section_ready("10.05", HAS_DASHSCOPE, "未检测到 DASHSCOPE_API_KEY"):
    child_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", "。", "；", "，", " "],
        chunk_size=120,
        chunk_overlap=30,
        add_start_index=True,
    )
    parent_splitter = RecursiveCharacterTextSplitter(
        separators=["\n\n\n", "\n\n", "\n", "。", "；", "，", " "],
        chunk_size=240,
        chunk_overlap=40,
        add_start_index=True,
    )

    parent_retriever = ParentDocumentRetriever(
        vectorstore=Chroma(
            collection_name=f"parent_docs_{uuid.uuid4().hex[:8]}",
            embedding_function=get_embeddings(),
            persist_directory=str(CACHE_DIR / "parent_chroma"),
        ),
        docstore=InMemoryStore(),
        child_splitter=child_splitter,
        parent_splitter=parent_splitter,
    )

    parent_retriever.add_documents(rag_raw_documents)
    parent_docs = parent_retriever.invoke("哪里可以了解高考成绩")
    print("返回文档数：", len(parent_docs))
    print(parent_docs[0].page_content[:180])

返回文档数： 3
一、在哪里可以了解高考成绩、志愿填报时间和方式、各高校招生计划、往年录取参考等志愿填报权威信息？
各省级教育行政部门或招生考试机构官方网站、微信公众号等权威渠道都会公布今年高考各阶段工作时间安排，包括高考成绩公布时间和查询方式、志愿填报时间，以及今年各高校招生计划、往年录取情况参考等权威信息。考生和家长要及时关注本地官方权威渠道发布的消息内容。


## 10.06 MultiVectorRetriever（假设问题版）

- 做法是先让模型给文档生成几个“适合检索的问题”，再拿这些问题建立子向量。

In [26]:
import re

from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_core.documents import Document
from langchain_core.exceptions import OutputParserException
from langchain_core.messages import AIMessage
from langchain_core.prompts import ChatPromptTemplate


def parse_questions(ai_message: AIMessage) -> list[str]:
    text = ai_message.content.strip()
    candidates = [item.strip() for item in text.split("\n\n") if item.strip()]
    if len(candidates) < 3:
        candidates = [item.strip() for item in text.split("\n") if item.strip()]
    candidates = [re.sub(r"^\d+[\.、\)]\s*", "", item) for item in candidates]
    candidates = [item for item in candidates if item]
    if len(candidates) < 3:
        raise OutputParserException(f"问题生成格式不符合预期：{ai_message.content}")
    return candidates[:3]


if section_ready("10.06", HAS_DASHSCOPE, "未检测到 DASHSCOPE_API_KEY"):
    sample_docs = rag_documents[:3]
    question_chain = (
        {"doc": lambda x: x.page_content}
        | ChatPromptTemplate.from_template(
            "请基于下面内容，生成 3 个适合检索的中文问题。\n\n{doc}\n\n要求：每个问题之间用两个换行分隔。"
        )
        | get_chat_model(temperature=0)
        | parse_questions
    )

    generated_questions = question_chain.batch(sample_docs, {"max_concurrency": 2})
    print("第一组问题：", generated_questions[0])

    id_key = "doc_id"
    doc_ids = [str(uuid.uuid4()) for _ in sample_docs]
    question_docs = []
    for index, questions in enumerate(generated_questions):
        for question in questions:
            question_docs.append(
                Document(
                    page_content=question,
                    metadata={id_key: doc_ids[index]},
                )
            )

    multi_vector_retriever = MultiVectorRetriever(
        vectorstore=Chroma(
            collection_name=f"multi_vector_{uuid.uuid4().hex[:8]}",
            embedding_function=get_embeddings(),
            persist_directory=str(CACHE_DIR / "multi_vector_chroma"),
        ),
        byte_store=InMemoryByteStore(),
        id_key=id_key,
    )
    multi_vector_retriever.vectorstore.add_documents(question_docs)
    multi_vector_retriever.docstore.mset(list(zip(doc_ids, sample_docs)))

    multi_vector_docs = multi_vector_retriever.invoke("哪里可以了解高考成绩")
    print("返回文档数：", len(multi_vector_docs))
    print(multi_vector_docs[0].page_content[:180])

第一组问题： ['2024年全国高考的报名人数是多少？比2023年增加了多少人？', '哪些省份是2024年首次实施新高考的第四批高考综合改革省份？', '2024年全国高考的全称是什么？其报考对象包括哪些人群？']


返回文档数： 3
2024年高考全国统考于2024年6月7日开始举行，部分省份考试时间为2天，实行新高考的省份为3-4天 [29]。5月31日，2024年高考试卷从北京发往全国 [22]。6月5日，2024年高考举报电话已开通，教育部教育考试院的举报电话为：010-62790357 [53] [77]。
